# 08 — Story Naming

## What this notebook does

**Part 1 — Global Centroid Naming (explored)**
Each story is named once using all its articles. Simple baseline, but produces stale names
for long-running stories as coverage shifts over time.

**Part 2 — Per-Day Centroid Naming (selected)**
Each story is named separately for each date it is active, using only the articles from that day's dots.
This keeps the name current as the story evolves.

## Result files
- `data/processed/story_names.pkl` — `{story_id: title}` — global approach (reference)
- `data/processed/story_names_per_day.pkl` — `{(story_id, date): title}` — selected approach

In [9]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

REPO_ROOT = Path("../").resolve()

# Load Data

In [ ]:
df = pd.read_parquet(REPO_ROOT / "data/processed/articles_clean.parquet")
df["published_at_dt"] = pd.to_datetime(df["published_at_dt"], utc=True)

embs_all = np.load(REPO_ROOT / "data/processed/embeddings.npy")

df = df.iloc[:len(embs_all)].reset_index(drop=True)

with open(REPO_ROOT / "data/processed/stories.pkl", "rb") as f:
    stories = pickle.load(f)

with open(REPO_ROOT / "data/processed/dots_louvain.pkl", "rb") as f:
    all_dots = pickle.load(f)

print(f"Articles  : {len(df):,}")
print(f"Embeddings: {embs_all.shape}")
print(f"Stories   : {len(stories):,}")
print(f"Date range: {df['published_at_dt'].min().date()} → {df['published_at_dt'].max().date()}")

Articles  : 30,462
Embeddings: (30462, 1024)
Stories   : 18,473
Date range: 2026-04-20 → 2026-05-22


# Part 1 — Global Centroid Naming (Explored)

For each story, compute the mean embedding of **all** its articles (the global centroid),
then select the article closest to it as the story name.

**Limitation**: for long-running stories, the global centroid reflects the story's entire history.
The name anchors to an early or middle moment and becomes stale as coverage evolves.
A story about Iran/US peace talks named on day 1 will keep that name even if the
story has shifted to ceasefires and sanctions weeks later.

In [3]:
def name_stories(stories: dict, embs_all, df) -> dict:
    story_names = {}
    for sid, arts in stories.items():
        if len(arts) == 0:
            continue
        embs = embs_all[arts]
        centroid = embs.mean(axis=0, keepdims=True)
        sims = cosine_similarity(centroid, embs)[0]
        best_idx = arts[sims.argmax()]
        story_names[sid] = df.iloc[best_idx]["title"]
    return story_names

In [4]:
story_names = name_stories(stories, embs_all, df)
print(f"Named {len(story_names):,} stories")

Named 18,473 stories


# Inspect Sample Results

In [5]:
def inspect_story(sid, stories, story_names, df):
    arts = stories.get(sid)
    if arts is None:
        print(f"Story {sid} not found.")
        return
    name = story_names.get(sid, "N/A")
    sorted_arts = sorted(arts, key=lambda i: df.iloc[i]["published_at_dt"])
    print(f"Story ID : {sid}")
    print(f"Articles : {len(arts)}")
    print(f"Name     : {name}")
    print("\nAll articles:")
    for idx in sorted_arts:
        pub = df.iloc[idx]["published_at_dt"].strftime("%d %b %H:%M")
        print(f"  - {pub} [{df.iloc[idx]['source']}] {df.iloc[idx]['title']}")

In [6]:
# inspect a medium-sized story (10-20 articles)
story_sizes = sorted(stories.items(), key=lambda x: -len(x[1]))
medium_stories = [(sid, arts) for sid, arts in story_sizes if 10 <= len(arts) <= 20]

inspect_story(medium_stories[0][0], stories, story_names, df)

Story ID : 593
Articles : 20
Name     : НА ЖИВО: Какво предвижда американското предложение за мир към Иран?

All articles:
  - 22 Apr 21:01 [actualno] НА ЖИВО: Тръмп е поставил ултиматум на Иран от 3 до 5 дни
  - 23 Apr 21:01 [actualno] НА ЖИВО: Тръмп пред край на войната в Иран - остава му малко време, преди Конгресът да може да го спре
  - 26 Apr 21:01 [actualno] НА ЖИВО: Помогнала ли е България на САЩ за войната в Иран?
  - 27 Apr 21:01 [actualno] НА ЖИВО: Иран потърси помощ от Русия след провалените преговори със САЩ
  - 28 Apr 21:01 [actualno] НА ЖИВО: Тръмп заговори за "колапс" на Иран, обмисля два варианта за войната
  - 29 Apr 21:12 [actualno] НА ЖИВО: Тръмп готви дългосрочна блокада на Ормузкия проток
  - 03 May 21:01 [actualno] НА ЖИВО: Иран изпрати на САЩ нов план за край на войната
  - 04 May 21:44 [actualno] НА ЖИВО: Размяна на удари на фона на примирието между САЩ и Иран
  - 06 May 21:39 [actualno] НА ЖИВО: Близо ли са до мир САЩ и Иран?
  - 07 May 21:11 [actualno] НА ЖИВ

In [7]:
inspect_story(medium_stories[1][0], stories, story_names, df)

Story ID : 5614
Articles : 20
Name     : САЩ ще изтеглят 5 000 войници от Германия

All articles:
  - 29 Apr 21:35 [actualno] Тръмп: Обмисляме намаляване на броя на американските войници в Германия
  - 30 Apr 01:44 [fakti] Вашингтон обмисля намаляване на броя на американските въоръжени сили в Германия
  - 30 Apr 03:15 [nova] След спор с Мерц: Тръмп обмисля намаляване на американските войници в Германия
  - 30 Apr 04:09 [24chasa] Тръмп обмисля намаляване на американските войници в Германия след спор с Мерц
  - 30 Apr 19:45 [24chasa] Тръмп плаши да изтегли 36 хиляди войници от Германия (Обзор)
  - 02 May 00:38 [fakti] САЩ изтеглят 5 000 войници от Германия
  - 02 May 05:07 [fakti] Нежеланието на Европа да ръководи НАТО доведе до намаляване на американските войски в Германия
  - 02 May 06:28 [24chasa] САЩ ще изтеглят 5000 американски военнослужещи от Германия до една година
  - 02 May 08:55 [actualno] Тръмп "наказва" Берлин заради Иран: Хиляди американски войници напускат Германия
  - 02 

## Save — Global Approach (reference artifact)

In [8]:
out_path = REPO_ROOT / "data/processed/story_names.pkl"
with open(out_path, "wb") as f:
    pickle.dump(story_names, f)

print(f"Saved {len(story_names):,} story names -> {out_path}")

Saved 18,473 story names -> /Users/ivanadonchevska/Projects/msc_thesis/data/processed/story_names.pkl


# Part 2 — Per-Day Centroid Naming (Selected)

For each story and each date it is active:
1. Collect all articles in that story's dots on that date (the **day window**)
2. Compute their mean BGE-M3 embedding — the **day centroid**
3. Find the article with the highest cosine similarity to the day centroid — the **day medoid**
4. Use that article's title as the story name for that date

This scopes the centroid to the active coverage window, so the name tracks *where the story currently is*
rather than its entire history. The key `(story_id, date)` allows the app to display a name that
reflects the day's developments.

In [10]:
def name_stories_per_day(all_dots: dict, embs_all, df) -> dict:
    """Returns {(story_id, date): title} for every (story, date) pair."""
    names = {}
    for sid, dots in all_dots.items():
        by_date = defaultdict(list)
        for dot in dots:
            dt = dot["effective_start"].date()
            by_date[dt].extend(dot["indices"])
        for dt, indices in by_date.items():
            embs = embs_all[indices]
            centroid = embs.mean(axis=0, keepdims=True)
            sims = cosine_similarity(centroid, embs)[0]
            best_idx = indices[int(sims.argmax())]
            names[(sid, dt)] = df.iloc[best_idx]["title"]
    return names

In [11]:
story_names_per_day = name_stories_per_day(all_dots, embs_all, df)
print(f"Named {len(story_names_per_day):,} (story, date) pairs")

Named 20,577 (story, date) pairs


## Inspect: Global vs Per-Day Names on a Long Story

For a long-running story, the global name is fixed to one title for the entire lifetime.
The per-day name tracks the story's current focus as coverage shifts day by day.

In [12]:
# Story 9660 — Hormuz Strait, 39 articles spanning 21 Apr → 22 May
sid = 9660

print(f"Global name: {story_names.get(sid)}")
print()
print("Per-day names:")
day_names = sorted(
    [(dt, title) for (s, dt), title in story_names_per_day.items() if s == sid]
)
for dt, title in day_names:
    print(f"  {dt}  {title}")

Global name: Европейски страни водят преговори с Иран за преминаването на Ормузкия проток

Per-day names:
  2026-04-21  Само три кораба са преминали през Ормузкия проток през последните 24 часа
  2026-04-23  Иран отчита първи приходи от такси в Ормузкия проток
  2026-04-24  Само 5 кораба са преминали през Ормузкия проток през последните 24 часа, сочат платформи за следене
  2026-04-27  Ирански депутат: Армията ни трябва да контролира Ормузкия проток
  2026-05-05  Иран въведе нов режим за преминаване на кораби през Ормузкия проток
  2026-05-07  Иранската агенция Фарс съобщи, че са били чути експлозии близо до град Бандар Абас
  2026-05-14  Иран: Около 30 кораба са минали през Ормузкия проток за последната нощ
  2026-05-16  Европейски страни водят преговори с Иран за преминаването на Ормузкия проток
  2026-05-18  Иран официално създаде орган за управление на Ормузкия проток
  2026-05-20  Иран пусна 26 кораба през Ормузкия проток
  2026-05-22  ЕС разширява санкциите срещу Иран заради дейс

## Save — Per-Day Approach

In [13]:
out_path = REPO_ROOT / "data/processed/story_names_per_day.pkl"
with open(out_path, "wb") as f:
    pickle.dump(story_names_per_day, f)

print(f"Saved {len(story_names_per_day):,} per-day story names → {out_path}")

Saved 20,577 per-day story names → /Users/ivanadonchevska/Projects/msc_thesis/data/processed/story_names_per_day.pkl
